# 03 — MWA Data & Imaging

Find MWA solar observations, download visibilities, preprocess, calibrate, and image with WSClean.

This notebook mirrors the [mwa-demo pipeline](https://github.com/i4Ds/mwa-demo) but targets the **sun** rather than a fixed sky position.

**Prerequisites:** `birli`, `hyperdrive`, `wsclean`, `giant-squid`, and an MWA ASVO API key in `$MWA_ASVO_API_KEY`.

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd
import pyvo
from astropy.time import Time, TimeDelta
from astropy.io import fits
import matplotlib.pyplot as plt

## 1. Find solar MWA observations via TAP

Query the [MWA VO TAP service](http://vo.mwatelescope.org/mwa_asvo/tap) for observations pointing near the sun.

Key difference from mwa-demo: `sun_elevation > 20` (daytime, sun reasonably high).

In [ ]:
# --- set your event window here ---
event_start = Time('2021-05-07 18:40:00', scale='utc')
event_end   = Time('2021-05-07 19:10:00', scale='utc')

# GPS times for the TAP query (MWA obs_id is GPS start time)
gps_start = int(event_start.gps)
gps_end   = int(event_end.gps)

# proprietary cutoff: 18 months ago
proprietary = int((Time.now() - TimeDelta(548, format='jd')).gps)

tap = pyvo.dal.TAPService('http://vo.mwatelescope.org/mwa_asvo/tap')

obs = (
    tap.search(f"""
        SELECT TOP 20 *
        FROM mwa.observation
        WHERE obs_id BETWEEN {gps_start} AND {gps_end}
        AND sun_elevation > 20
        AND obs_id < {proprietary}
        AND good_tiles >= 112
        AND dataquality <= 1
        AND deleted_flag != 'TRUE'
        AND gpubox_files_archived > 1
        ORDER BY obs_id DESC
    """)
    .to_table()
    .to_pandas()
    .dropna(axis=1, how='all')
)

obs['config']    = obs['mwa_array_configuration'].str.split(' ').str[-1]
obs['gigabytes'] = obs['total_archived_data_bytes'] / 1e9

print(f"{len(obs)} observations found")
obs[['obs_id', 'starttime_utc', 'obsname', 'config', 'sun_elevation', 'gigabytes']]

## 2. Select an observation and set up paths

In [ ]:
obsid    = int(obs['obs_id'].iloc[0])   # pick first result, or set manually
data_dir = Path('../data') / str(obsid)

for sub in ('raw', 'prep', 'cal', 'img'):
    (data_dir / sub).mkdir(parents=True, exist_ok=True)

print(f"obsid    : {obsid}")
print(f"data_dir : {data_dir.resolve()}")

## 3. Download via MWA ASVO (giant-squid)

Submit a raw visibility download job and wait for it to complete.  
Run the cells below in order; the download can take 10–60 min depending on queue.

In [ ]:
# Submit raw visibility job
result = subprocess.run(
    ['giant-squid', 'submit-vis', str(obsid)],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)

In [ ]:
# Check job status (run repeatedly until 'Ready')
result = subprocess.run(
    ['giant-squid', 'list', str(obsid)],
    capture_output=True, text=True,
)
print(result.stdout)

In [ ]:
# Download when ready  — replace JOB_ID with the ID from giant-squid list
JOB_ID = None   # e.g. 12345

if JOB_ID:
    result = subprocess.run(
        ['giant-squid', 'download', str(JOB_ID), '--keep-zip', '-o', str(data_dir / 'raw')],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)

## 4. Inspect metadata with mwalib

In [ ]:
import mwalib

metafits_path = next(data_dir.glob('raw/*.metafits'), None)
print(f"metafits: {metafits_path}")

if metafits_path:
    ctx = mwalib.MetafitsContext(str(metafits_path))
    print(f"Obs name      : {ctx.obs_name}")
    print(f"Start UTC     : {ctx.sched_start_utc}")
    print(f"Duration      : {ctx.obs_bandwidth_hz/1e6:.1f} MHz bandwidth")
    print(f"Antennas      : {ctx.num_ants}")
    print(f"Timesteps     : {ctx.num_timesteps}")

## 5. Preprocess with Birli

Flags, averages, and outputs a preprocessed uvfits file.

In [ ]:
metafits = str(metafits_path)
raw_gpufits = sorted((data_dir / 'raw').glob('*.fits'))   # gpubox files
prep_uvfits = str(data_dir / 'prep' / f'birli_{obsid}.uvfits')

birli_cmd = [
    'birli',
    '-m', metafits,
    '--uvfits-out', prep_uvfits,
    '--avg-freq-res', '40',
    '--avg-time-res', '2',
    '--flag-edge-width', '80',
] + [str(f) for f in raw_gpufits]

print('Running:', ' '.join(birli_cmd))
# result = subprocess.run(birli_cmd, capture_output=True, text=True)
# print(result.stdout); print(result.stderr)

## 6. Calibrate with Hyperdrive

In [ ]:
srclist   = Path('../data/srclist_pumav3_EoR0LoBES_EoR1pietro_CenA-GP_2023-11-07.yaml')
cal_sol   = str(data_dir / 'cal' / f'hyp_soln_{obsid}.fits')
cal_ms    = str(data_dir / 'cal' / f'hyp_cal_{obsid}.ms')

hyp_cmd = [
    'hyperdrive', 'di-calibrate',
    '-s', str(srclist),
    '-d', prep_uvfits, metafits,
    '--outputs', cal_sol,
]

print('Running:', ' '.join(hyp_cmd))
# result = subprocess.run(hyp_cmd, capture_output=True, text=True)
# print(result.stdout); print(result.stderr)

## 7. Image with WSClean

In [ ]:
import os

beam_file = Path(os.environ.get('MWA_BEAM_FILE', '../data/mwa_full_embedded_element_pattern.h5'))
beam_path = str(beam_file.parent)
imgname   = str(data_dir / 'img' / f'wsclean_{obsid}')

wsclean_cmd = [
    'wsclean',
    '-name', imgname,
    '-size', '2048', '2048',
    '-scale', '20asec',
    '-pol', 'i',
    '-niter', '100',
    '-gridder', 'idg', '-grid-with-beam', '-idg-mode', 'cpu',
    '-multiscale',
    '-weight', 'briggs', '0',
    '-mgain', '0.85', '-gain', '0.1',
    '-auto-threshold', '1', '-auto-mask', '3',
    '-no-update-model-required',
    '-make-psf',
    '-small-inversion',
    '-mwa-path', beam_path,
    cal_ms,
]

print('Running:', ' '.join(wsclean_cmd))
# result = subprocess.run(wsclean_cmd, capture_output=True, text=True)
# print(result.stdout); print(result.stderr)

## 8. Display the image

In [ ]:
image_fits = Path(f'{imgname}-image.fits')

if image_fits.exists():
    with fits.open(image_fits) as hdul:
        data = hdul[0].data.squeeze()

    vmax = data[data > 0].std() * 5
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(data, origin='lower', cmap='inferno', vmin=-vmax/5, vmax=vmax)
    plt.colorbar(im, ax=ax, label='Jy/beam')
    ax.set_title(f'MWA image — obsid {obsid}')
    plt.tight_layout()
    plt.show()
else:
    print(f'Image not found: {image_fits}')